In [1]:
import os
from calendar import monthrange

dir="/home/songyl/GitHub/pheno_tweet/"
category="pollen"
year="2021"
keywords = open(os.path.join(dir,"script/keywords/",category+".txt"), "r").read().split("\n")

month_list=[str(i).zfill(2) for i in range(1,13)]
for month in month_list:
    day_list=[str(i).zfill(2) for i in range(1,monthrange(int(year), int(month))[1]+1)]
    for day in day_list:
        path=os.path.join(dir,"data/query/",category+"/","Spark/",year+"/",month+"/", day+"/")
        if os.path.isdir(path):
            df = sqlContext.read.json(path)
            df_sel = df.select('created_at','user.id','user.screen_name','user.description','extended_tweet.full_text','lang') # screen_name appears to be the handle
            df_sel = df_sel.filter(df_sel['full_text'].isNotNull())
            df_lang = df_sel.filter(df_sel['lang']=="en")
            df_word = None
            for keyword in keywords:
                df_word_one = df_lang.filter(df_lang['full_text'].rlike('\\b(?i)'+keyword+'\\b')) # word boundary # case insensitive
                if not df_word:
                    df_word = df_word_one
                else:
                    df_word = df_word.union(df_word_one)
            df_word=df_word.distinct()
            df_word.write.option("header",True).option("delimiter","\t").mode("overwrite").csv(os.path.join(dir,"data/query/",category+"/","CSV/",year+"/",month+"/", day+"/"))


22/12/02 09:31:21 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
